# Deepfake Detection Training - Local Development
This notebook has been refactored to run locally in VS Code. Google Colab and Google Drive dependencies have been removed.

In [8]:
import os
import sys
import torch
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

# Add backend to path so we can import the model
BASE_DIR = Path(os.getcwd()).parents[1]
sys.path.append(str(BASE_DIR / "backend"))

from vit_model import ViTDeepfakeDetector

print(f"Project Base Directory: {BASE_DIR}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Project Base Directory: d:\deepfake detection project\Deepfake-Detection-Multimodal-Explainable-AI-Mini-project
Using device: cpu


## Dataset Configuration

In [9]:
DATASET_DIR = Path("datasets/real_and_fake_face")

def get_video_files():
    # Note: Adjust these paths if your videos are stored elsewhere
    # This script looks for videos in the datasets folder
    fake_videos = glob.glob(str(DATASET_DIR / "training_fake" / "*.mp4"))
    real_videos = glob.glob(str(DATASET_DIR / "training_real" / "*.mp4"))
    
    # If no mp4s found, check for the image-based dataset provided in the repo
    if not fake_videos and not real_videos:
        print("No mp4 videos found, checking for images...")
        fake_images = glob.glob(str(DATASET_DIR / "training_fake" / "*.jpg"))
        real_images = glob.glob(str(DATASET_DIR / "training_real" / "*.jpg"))
        return real_images + fake_images
        
    return real_videos + fake_videos

video_files = get_video_files()
print(f"Total files found: {len(video_files)}")

No mp4 videos found, checking for images...
Total files found: 2041


In [10]:
class LocalDeepfakeDataset(Dataset):
    def __init__(self, file_paths, sequence_length=10, transform=None):
        self.file_paths = file_paths
        self.sequence_length = sequence_length
        self.transform = transform
        
    def __len__(self):
        return len(self.file_paths)
        
    def __getitem__(self, idx):
        path = self.file_paths[idx]
        # Label mapping: REAL=0, FAKE=1 (consistent with backend)
        label = 1 if "fake" in path.lower() else 0
        
        frames = []
        if path.endswith(".mp4"):
            cap = cv2.VideoCapture(path)
            while len(frames) < self.sequence_length:
                ret, frame = cap.read()
                if not ret: break
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if self.transform: frame = self.transform(frame)
                frames.append(frame)
            cap.release()
        else:
            # It's an image, repeat it to simulate a sequence
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            if self.transform: img = self.transform(img)
            for _ in range(self.sequence_length):
                frames.append(img)
                
        # Pad if not enough frames
        while len(frames) < self.sequence_length:
            frames.append(frames[-1] if frames else torch.zeros(3, 224, 224))
            
        return torch.stack(frames), label

im_size = 224 # Synchronized with ViT backend
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((im_size, im_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Model Initialization

In [11]:
# Initialize model with synchronized parameters
model = ViTDeepfakeDetector(
    img_size=224,
    patch_size=16,
    embed_dim=192,
    depth=3,
    num_heads=3
).to(device)

print("ViT Model Initialized")

ViT Model Initialized


## Training Loop

In [14]:
import os
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

# Simple split
split = int(0.8 * len(video_files))
train_files = video_files[:split]
test_files = video_files[split:]

train_dataset = LocalDeepfakeDataset(train_files, sequence_length=5, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

num_epochs = 2
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for i, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        if i % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{i}/{len(train_loader)}], Loss: {loss.item():.4f}")

    MODEL_SAVE_PATH = os.path.join(str(BASE_DIR), "backend", "model.pth")
    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"Model saved to {MODEL_SAVE_PATH}")


Epoch [1/2], Batch [0/408], Loss: 0.6031
Epoch [1/2], Batch [10/408], Loss: 0.1641
Epoch [1/2], Batch [20/408], Loss: 0.0576
Epoch [1/2], Batch [30/408], Loss: 0.0185
Epoch [1/2], Batch [40/408], Loss: 0.0074
Epoch [1/2], Batch [50/408], Loss: 0.0041
Epoch [1/2], Batch [60/408], Loss: 0.0024
Epoch [1/2], Batch [70/408], Loss: 0.0013
Epoch [1/2], Batch [80/408], Loss: 0.0010
Epoch [1/2], Batch [90/408], Loss: 0.0008
Epoch [1/2], Batch [100/408], Loss: 0.0006
Epoch [1/2], Batch [110/408], Loss: 0.0005
Epoch [1/2], Batch [120/408], Loss: 0.0002
Epoch [1/2], Batch [130/408], Loss: 0.0002
Epoch [1/2], Batch [140/408], Loss: 0.0002
Epoch [1/2], Batch [150/408], Loss: 0.0001
Epoch [1/2], Batch [160/408], Loss: 0.0000
Epoch [1/2], Batch [170/408], Loss: 0.0001


KeyboardInterrupt: 

In [15]:
torch.save(model.state_dict(), "../../backend/model.pth")
print("Model saved successfully!")


Model saved successfully!
